In [1]:
from datasets import load_dataset

ds = load_dataset("JacobLinCool/VoiceBank-DEMAND-16k")

/home/ys/diploma/denoising_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import torch

class TestDataset(torch.utils.data.Dataset):
    def __init__(self,ds):
        super().__init__()
        self.ds = ds

    def __getitem__(self, index):
        
        return torch.tensor(self.ds[index]['clean']['array']).to(torch.float32), torch.tensor(self.ds[index]['noisy']['array']).to(torch.float32)
    
    def __len__(self):
        return self.ds.__len__()
    

test_set = TestDataset(ds['test'].select(range(100)))
test_loader = torch.utils.data.DataLoader(test_set, batch_size=1, )
a = next(iter(test_loader))[0]
a.repeat((2, 1)).shape

In [5]:
import cpuinfo
from time import perf_counter
from loaders import *
from lightning_module import *
from pipeline.cfg_loader import load_cfg
import GPUtil
from torchmetrics.audio import( ScaleInvariantSignalDistortionRatio as SISDR, SignalDistortionRatio as SDR,
                                SignalNoiseRatio as SNR, ScaleInvariantSignalNoiseRatio as SISNR,
                                PerceptualEvaluationSpeechQuality as PESQ,
                                ShortTimeObjectiveIntelligibility as STOI)



cfg_path ='configs/denoise_model_v1_cfg.yaml'
ckpt_path = 'configs/last.ckpt'
model_cfg = load_cfg(cfg_path)

metrics = dict( pesq_wb = PESQ(16000, 'wb'),
                pesq_nb = PESQ(16000, 'nb'),
                stoi = STOI(16000),
                snratio = SNR(),
                sdratio = SDR(),
                sisdratio = SISDR(),
                sisnratio = SISNR())
                

model = SpectrogramLightningModelUnet.load_from_checkpoint(ckpt_path,
                                                            **model_cfg)
info = cpuinfo.get_cpu_info()
gpus = GPUtil.getGPUs()
print(" "*50)
print("-"*20 + 'DEVICE INFO' + "-"*20)
print(f"Процессор: {info['brand_raw']}")
print(f"Количество ядер: {info['count']}")
for g in gpus:
    print(f"Видеокарта: {g.name}")
    print(f"Память: {g.memoryTotal} MB")
    print(f"Используется памяти: {g.memoryUsed} MB")
    print(f"Загрузка GPU: {g.load * 100}%")
print("-"*20 + '----------' + "-"*20)   
    
def compute_metrics(cleaned_wf, clean_wf, metric):
        cleaned_wf = cleaned_wf.detach().cpu()
        clean_wf = clean_wf.detach().cpu()
        cleaned_wf_shape = cleaned_wf.shape[-1]
        clean_wf_shape = clean_wf.shape[-1]
        if cleaned_wf.shape[1] != 1:
            cleaned_wf = cleaned_wf.sum(1, keepdims=True)
        if clean_wf.shape[1] != 1:
            clean_wf = clean_wf.sum(1, keepdims=True)
        if clean_wf_shape == min(clean_wf_shape, cleaned_wf_shape):
            cleaned_wf = cleaned_wf[:, :, :clean_wf_shape]
        else:
            clean_wf = clean_wf[:, :, :cleaned_wf_shape]

        metric['pesq_wb'].update(cleaned_wf, clean_wf)
        metric['pesq_nb'].update(cleaned_wf, clean_wf)
        metric['stoi'].update(cleaned_wf, clean_wf)

        metric['snratio'].update(cleaned_wf, clean_wf)
        metric['sisdratio'].update(cleaned_wf, clean_wf)
        metric['sdratio'].update(cleaned_wf, clean_wf)
        metric['sisnratio'].update(cleaned_wf, clean_wf)
        
def bench_model(model,metrics, loader, num_iters=100):
    model.eval()
    cpu = []
    gpu = []
    for i, batch in enumerate(loader):
        
        model = model.to('cpu')
        mixed, clean = batch
        
        mixed = mixed.to('cpu')
        
        start = perf_counter()
        out = model.run(mixed)
        delta =perf_counter() - start
        cpu.append(delta)
        
        model = model.to('cuda')
        mixed = mixed.to('cuda')
        clean = clean.to('cuda')
        
        
        start = perf_counter()
        out = model.run(mixed)
        delta = perf_counter() - start
        gpu.append(delta)

        out = torch.tensor(out).to('cuda')

        if out.shape[-1] > clean.shape[-1]:
            out = out[..., :clean.shape[-1]]
        
        if clean.shape[0] == 1 and clean.ndim == 2:
            clean = clean.repeat((2, 1))
        if clean.ndim != out.ndim:
            clean = clean[None, ...]
       
        compute_metrics(out, clean, metrics)
        
        if (i + 1) % (num_iters) == 0:
            break

    
    snratio = metrics['snratio'].compute()
    sisdratio = metrics['sisdratio'].compute()
    sdratio = metrics['sdratio'].compute()
    sisnratio = metrics['sisnratio'].compute()
    pesq_wb = metrics['pesq_wb'].compute()
    pesq_nb = metrics['pesq_nb'].compute()
    stoi= metrics['stoi'].compute()

    metric_values = dict(stoi = stoi.item(),
                        pesqnb = pesq_nb.item(),
                        pesqwb = pesq_wb.item(),
                        snr = snratio.item(),
                        sisdr = sisdratio.item(),
                        sdr = sdratio.item(),
                        sisnr = sisnratio.item())

    return cpu, gpu, metric_values


model


                                                  
--------------------DEVICE INFO--------------------
Процессор: Intel(R) Core(TM) i5-10300H CPU @ 2.50GHz
Количество ядер: 8
Видеокарта: NVIDIA GeForce GTX 1650
Память: 4096.0 MB
Используется памяти: 295.0 MB
Загрузка GPU: 0.0%
--------------------------------------------------


SpectrogramLightningModelUnet(
  (stft): Spectrogram()
  (model): DenoisingModelUnet(
    (encoder): SpectrumEncoder(
      (encoder_features): Sequential(
        (layer_0): AdaptiveResBlock(
          (block): MobileBlock(
            (conv): Conv2d(2, 8, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4))
            (depth_wise): Conv2d(8, 8, kernel_size=(9, 9), stride=(1, 1), groups=8)
            (point_wise): Conv2d(8, 8, kernel_size=(1, 1), stride=(1, 1), padding=(4, 4))
            (bn): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (act): ELU(alpha=1.0)
            (dropout): Dropout2d(p=0.2, inplace=False)
          )
          (scale): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
          (adapt_res): Conv2d(2, 8, kernel_size=(1, 1), stride=(1, 1), bias=False)
        )
        (layer_1): AdaptiveResBlock(
          (block): MobileBlock(
            (conv): Conv2d(8, 32, kernel_size=(7, 7), stride=(1

# VoiceBank + DEMAND

In [ ]:
cpu, gpu,metric_values = bench_model(model, metrics, loader=test_loader)


print('-'*20 + '+CPU+' + '-'*20)
print('Avg CPU Inference [s] : ', torch.tensor(cpu).mean().item())
print('-'*20 + '-----' + '-'*20)
print('-'*20 + '+GPU+' + '-'*20)
print('Avg GPU Inference [s] : ', torch.tensor(gpu).mean().item())
print('-'*20 + '-----' + '-'*20)
print('-'*19 + 'METRICS' + '-'*19 )
print(f"SNR [dB]: {metric_values['snr']}"),
print(f"SDR [dB]: {metric_values['sdr']}")
print(f"SI-SDR [dB]: {metric_values['sisdr']}")
print(f"SI-SNR [dB]: {metric_values['sisnr']}")
print(f"STOI: {metric_values['stoi']}")
print(f"PESQ-NB: {metric_values['pesqnb']}")
print(f"PESQ-WB: {metric_values['pesqwb']}")

print('-'*19 + '-------' + '-'*19 )
print(" "*50)

--------------------+CPU+--------------------
Avg CPU Inference [s] :  1.048991084098816
---------------------------------------------
--------------------+GPU+--------------------
Avg GPU Inference [s] :  0.26157259941101074
---------------------------------------------
-------------------METRICS-------------------
SNR [dB]: 7.859231472015381
SDR [dB]: 10.862805366516113
SI-SDR [dB]: 6.800779342651367
SI-SNR [dB]: 6.801410675048828
STOI: 0.8085654377937317
PESQ-NB: 2.3817174434661865
PESQ-WB: 1.7474521398544312
---------------------------------------------
                                                  


# LibreSpeech + Wham

In [6]:
_, _, test_loader = get_loaders(speech_dirs=["dev-clean", "test-clean"],
                                                    noise_dir="./wham_noise//wham_noise",
                                                    batch_size=1,
                                                    padding_strategy=None)

#print(next(iter(test_loader))[0].shape)
cpu, gpu,metric_values = bench_model(model, metrics, loader=test_loader)


print('-'*20 + '+CPU+' + '-'*20)
print('Avg CPU Inference [s] : ', torch.tensor(cpu).mean().item())
print('-'*20 + '-----' + '-'*20)
print('-'*20 + '+GPU+' + '-'*20)
print('Avg GPU Inference [s] : ', torch.tensor(gpu).mean().item())
print('-'*20 + '-----' + '-'*20)
print('-'*19 + 'METRICS' + '-'*19 )
print(f"SNR [dB]: {metric_values['snr']}"),
print(f"SDR [dB]: {metric_values['sdr']}")
print(f"SI-SDR [dB]: {metric_values['sisdr']}")
print(f"SI-SNR [dB]: {metric_values['sisnr']}")
print(f"STOI: {metric_values['stoi']}")
print(f"PESQ-NB: {metric_values['pesqnb']}")
print(f"PESQ-WB: {metric_values['pesqwb']}")

print('-'*19 + '-------' + '-'*19 )
print(" "*50)

--------------------+CPU+--------------------
Avg CPU Inference [s] :  0.9736919403076172
---------------------------------------------
--------------------+GPU+--------------------
Avg GPU Inference [s] :  0.2426769882440567
---------------------------------------------
-------------------METRICS-------------------
SNR [dB]: 0.37545520067214966
SDR [dB]: 10.328632354736328
SI-SDR [dB]: 10.01400375366211
SI-SNR [dB]: 10.01632308959961
STOI: 0.901713490486145
PESQ-NB: 2.5331625938415527
PESQ-WB: 1.8842942714691162
---------------------------------------------
                                                  
